## Deep Research Agent — LangGraph

Agentic pipeline built with **LangGraph** and open-source models (Groq / Cerebras / OpenRouter).

Pipeline:
1. **Clarifier** — generates 3 scoping questions; user picks one and adds context
2. **Planner** — turns query + clarification into prioritised search terms
3. **Searcher** — runs all searches in parallel via DuckDuckGo (no API key needed)
4. **Sufficiency checker** — decides if evidence is enough; triggers up to 2 extra rounds
5. **Writer** — produces a structured Markdown report (1000+ words)
6. **Evaluator** — scores 0-10; approves or requests one revision
7. **Emailer** — sends the final report via SendGrid

Provider switchable via `RESEARCH_PROVIDER` env var: `groq` | `cerebras` | `openrouter` | `openai`

Tracing via LangSmith (set `LANGCHAIN_API_KEY` in `.env` — uses no inference tokens).

In [ ]:
!uv pip install langgraph langchain-groq langchain-openai langchain-community ddgs sendgrid gradio python-dotenv pydantic

In [ ]:
from __future__ import annotations
import asyncio
import os
import uuid
from typing import Annotated, Any, Optional
from typing_extensions import TypedDict

import sendgrid
from ddgs import DDGS
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Content, Email, Mail, To

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

from IPython.display import Image, display
import gradio as gr

In [ ]:
load_dotenv(override=True)

PROVIDER = os.environ.get("RESEARCH_PROVIDER", "groq")

_PROVIDERS: dict[str, dict] = {
    "groq":       {"model": "llama-3.3-70b-versatile",           "base_url": "https://api.groq.com/openai/v1",       "api_key_env": "GROQ_API_KEY"},
    "cerebras":   {"model": "qwen-3-235b-a22b-instruct-2507",    "base_url": "https://api.cerebras.ai/v1",          "api_key_env": "CEREBRAS_API_KEY"},
    "openrouter": {"model": "meta-llama/llama-3.3-70b-instruct", "base_url": "https://openrouter.ai/api/v1",       "api_key_env": "OPENROUTER_API_KEY"},
    "openai":     {"model": "gpt-4o-mini",                       "base_url": "https://api.openai.com/v1",           "api_key_env": "OPENAI_API_KEY"},
}

_cfg = _PROVIDERS.get(PROVIDER, _PROVIDERS["groq"])

def _build_llm(temperature: float = 0.3) -> ChatOpenAI:
    """Return a ChatOpenAI-compatible client for the configured provider."""
    return ChatOpenAI(
        model=_cfg["model"],
        base_url=_cfg["base_url"],
        api_key=os.environ.get(_cfg["api_key_env"], ""),
        temperature=temperature,
    )

# LangSmith tracing — uses LANGCHAIN_API_KEY, consumes no inference tokens
if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "deep-research-langgraph"
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"

print(f"Provider: {PROVIDER} | Model: {_cfg['model']}")

In [ ]:
# Pydantic schemas for structured LLM outputs

class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(description="Three concise clarifying questions to focus the research scope.")
    context_summary: str = Field(description="One-sentence summary of what is already understood from the query.")


class WebSearchPlan(BaseModel):
    queries: list[str] = Field(description="Five to eight prioritised web search terms, most important first.")


class ResearchSufficiency(BaseModel):
    is_sufficient: bool = Field(description="True if current evidence is enough for a thorough report.")
    additional_queries: list[str] = Field(default_factory=list, description="New search terms to fill gaps.")
    reasoning: str = Field(description="Brief explanation of the decision.")


class ReportEvaluation(BaseModel):
    score: float = Field(ge=0, le=10, description="Quality score 0-10.")
    is_approved: bool = Field(description="True if score >= 7 and report meets quality standards.")
    feedback: str = Field(description="Evaluation feedback.")
    suggestions: list[str] = Field(description="Actionable improvement suggestions if not approved.")


# LangGraph shared state — the whiteboard every node reads from and writes to
class ResearchState(TypedDict):
    query: str                         # original user query
    clarifying_questions: list[str]    # 3 questions from clarifier
    context_summary: str               # one-line summary from clarifier
    user_clarification: str            # user's selected/edited clarification
    search_plan: list[str]             # search terms from planner
    search_results: str                # accumulated search result text
    search_retries: int                # extra search rounds triggered by sufficiency agent
    is_sufficient: bool                # sufficiency verdict (read by router)
    report: str                        # markdown report from writer
    report_score: float                # evaluator score
    report_feedback: str               # evaluator feedback (fed back to writer on retry)
    report_retries: int                # revision rounds triggered by evaluator
    report_approved: bool              # evaluator verdict (read by router)
    email_status: str                  # sendgrid delivery status

In [ ]:
# Tools

@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo. Returns top 5 results with title, snippet and URL.
    Works with any provider — no OpenAI key required."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No results found for: {query}"
        return "\n\n---\n\n".join(
            f"{r.get('title', '')}\n{r.get('body', '')}\nSource: {r.get('href', '')}"
            for r in results
        )
    except Exception as exc:
        return f"Search error for '{query}': {exc}"


def send_report_email(subject: str, body: str) -> str:
    """Send the research report as an email via SendGrid.
    Reads SENDGRID_API_KEY, SENDGRID_FROM_EMAIL, SENDGRID_TO_EMAIL from env."""
    api_key   = os.environ.get("SENDGRID_API_KEY")
    from_addr = os.environ.get("SENDGRID_FROM_EMAIL", "sender@example.com")
    to_addr   = os.environ.get("SENDGRID_TO_EMAIL",   "recipient@example.com")
    if not api_key:
        return "error: SENDGRID_API_KEY not set"
    try:
        sg   = sendgrid.SendGridAPIClient(api_key=api_key)
        mail = Mail(
            Email(from_addr), To(to_addr), subject,
            Content("text/html", body.replace("\n", "<br>"))
        )
        res = sg.client.mail.send.post(request_body=mail.get())
        return f"sent (status {res.status_code})"
    except Exception as exc:
        return f"error: {exc}"

In [ ]:
# LLM instances — one per role so temperatures can differ

default_llm  = _build_llm(temperature=0.3)
creative_llm = _build_llm(temperature=0.6)

# with_structured_output uses tool-calling under the hood (not json_schema)
# so it works with Groq, Cerebras and OpenRouter
clarifier_llm  = default_llm.with_structured_output(ClarifyingQuestions)
planner_llm    = default_llm.with_structured_output(WebSearchPlan)
sufficiency_llm = default_llm.with_structured_output(ResearchSufficiency)
evaluator_llm  = default_llm.with_structured_output(ReportEvaluation)

In [ ]:
# Retry limits
MAX_SEARCH_RETRIES = 2
MAX_REPORT_RETRIES = 1


# ── Node functions ─────────────────────────────────────────────────────────────

def clarifier_node(state: ResearchState) -> dict:
    """Generate 3 scoping questions and a context summary for the query."""
    result = clarifier_llm.invoke([
        SystemMessage("Generate exactly 3 clarifying questions to focus a research query, and a one-sentence context summary."),
        HumanMessage(state["query"]),
    ])
    return {
        "clarifying_questions": result.questions[:3],
        "context_summary": result.context_summary,
    }


def planner_node(state: ResearchState) -> dict:
    """Turn the query + user clarification into a prioritised list of search terms."""
    extra = ""
    if state.get("search_retries", 0) > 0:
        extra = f"\nPrevious searches were insufficient. Focus on: {state.get('search_plan', [])}"
    result = planner_llm.invoke([
        SystemMessage("You are a research planner. Produce 5-8 prioritised web search terms, most important first."),
        HumanMessage(
            f"Research query: {state['query']}\n"
            f"User clarification: {state.get('user_clarification', '')}\n"
            f"{extra}"
        ),
    ])
    return {"search_plan": result.queries}


async def searcher_node(state: ResearchState) -> dict:
    """Run all planned searches in parallel using DuckDuckGo."""
    async def _one(q: str) -> str:
        try:
            result = await asyncio.to_thread(web_search.invoke, q)
            return f"### {q}\n{result}\n"
        except Exception as exc:
            return f"### {q}\nFailed: {exc}\n"

    parts = await asyncio.gather(*[_one(q) for q in state["search_plan"][:8]])
    new_results = "\n---\n".join(parts)
    existing = state.get("search_results", "")
    return {"search_results": (existing + "\n\n" + new_results).strip()}


def sufficiency_node(state: ResearchState) -> dict:
    """Decide if accumulated search results are enough for a thorough report."""
    result = sufficiency_llm.invoke([
        SystemMessage("Assess whether the research evidence is sufficient for a thorough report on the query."),
        HumanMessage(
            f"Query: {state['query']}\n"
            f"Clarification: {state.get('user_clarification', '')}\n\n"
            f"Search results (truncated):\n{state.get('search_results', '')[:5000]}"
        ),
    ])
    retries = state.get("search_retries", 0)
    return {
        "is_sufficient": result.is_sufficient,
        "search_retries": retries + (0 if result.is_sufficient else 1),
        # If not sufficient, overwrite search_plan with additional queries
        "search_plan": result.additional_queries if not result.is_sufficient else state["search_plan"],
    }


def writer_node(state: ResearchState) -> dict:
    """Write a detailed Markdown research report from the accumulated search results."""
    feedback_block = ""
    if state.get("report_feedback"):
        feedback_block = (
            f"\nYour previous report was rejected. Evaluator feedback:\n{state['report_feedback']}\n"
            "Please address all feedback points in this revision."
        )
    response = creative_llm.invoke([
        SystemMessage(
            "You are a senior research writer. Produce a detailed Markdown report "
            "(minimum 1000 words) with: executive summary, background, key findings, "
            "analysis, conclusions, and follow-up questions. Cite sources inline."
        ),
        HumanMessage(
            f"Query: {state['query']}\n"
            f"Clarification: {state.get('user_clarification', '')}\n"
            f"{feedback_block}\n\n"
            f"Research evidence:\n{state.get('search_results', '')[:8000]}"
        ),
    ])
    return {"report": response.content}


def evaluator_node(state: ResearchState) -> dict:
    """Score the report 0-10. Approve if score >= 7 and it addresses the query fully."""
    result = evaluator_llm.invoke([
        SystemMessage(
            "Score the research report 0-10. Approve (is_approved=True) only if: "
            "score >= 7, report is at least 800 words, fully addresses the query, "
            "and cites sources."
        ),
        HumanMessage(
            f"Query: {state['query']}\n"
            f"Success criteria: clear, comprehensive, sourced report\n\n"
            f"Report:\n{state.get('report', '')}"
        ),
    ])
    retries = state.get("report_retries", 0)
    return {
        "report_score": result.score,
        "report_feedback": result.feedback,
        "report_approved": result.is_approved,
        "report_retries": retries + (0 if result.is_approved else 1),
    }


def emailer_node(state: ResearchState) -> dict:
    """Send the approved report via SendGrid."""
    subject = f"Research Report: {state['query'][:60]}"
    status  = send_report_email(subject, state.get("report", ""))
    return {"email_status": status}


# ── Routing functions ──────────────────────────────────────────────────────────

def route_sufficiency(state: ResearchState) -> str:
    """If evidence is insufficient and retries remain, re-plan; otherwise write."""
    if state.get("is_sufficient") or state.get("search_retries", 0) >= MAX_SEARCH_RETRIES:
        return "writer"
    return "planner"


def route_evaluation(state: ResearchState) -> str:
    """If approved or retries exhausted, email; otherwise revise."""
    if state.get("report_approved") or state.get("report_retries", 0) >= MAX_REPORT_RETRIES:
        return "emailer"
    return "writer"

In [ ]:
# ── Build two graphs ───────────────────────────────────────────────────────────

# Graph 1: Clarifier only — runs once to generate scoping questions
clarifier_builder = StateGraph(ResearchState)
clarifier_builder.add_node("clarifier", clarifier_node)
clarifier_builder.add_edge(START, "clarifier")
clarifier_builder.add_edge("clarifier", END)
clarifier_graph = clarifier_builder.compile()

# Graph 2: Full research pipeline (planner → searcher → sufficiency → writer → evaluator → emailer)
research_builder = StateGraph(ResearchState)
research_builder.add_node("planner",     planner_node)
research_builder.add_node("searcher",    searcher_node)
research_builder.add_node("sufficiency", sufficiency_node)
research_builder.add_node("writer",      writer_node)
research_builder.add_node("evaluator",   evaluator_node)
research_builder.add_node("emailer",     emailer_node)

research_builder.add_edge(START,        "planner")
research_builder.add_edge("planner",    "searcher")
research_builder.add_edge("searcher",   "sufficiency")
research_builder.add_conditional_edges("sufficiency", route_sufficiency, {"writer": "writer", "planner": "planner"})
research_builder.add_edge("writer",     "evaluator")
research_builder.add_conditional_edges("evaluator",   route_evaluation,  {"emailer": "emailer", "writer": "writer"})
research_builder.add_edge("emailer",    END)

research_graph = research_builder.compile(checkpointer=MemorySaver())

# Visualise the research graph
display(Image(research_graph.get_graph().draw_mermaid_png()))

In [ ]:
import nest_asyncio
nest_asyncio.apply()


def _initial_state(query: str, user_clarification: str = "") -> ResearchState:
    """Return a fully initialised state dict for a research run."""
    return ResearchState(
        query=query,
        clarifying_questions=[],
        context_summary="",
        user_clarification=user_clarification,
        search_plan=[],
        search_results="",
        search_retries=0,
        is_sufficient=False,
        report="",
        report_score=0.0,
        report_feedback="",
        report_retries=0,
        report_approved=False,
        email_status="",
    )


async def run_clarifier(query: str) -> tuple[list[str], str]:
    """Phase 1: generate scoping questions. Returns (questions, context_summary)."""
    state = _initial_state(query)
    result = await clarifier_graph.ainvoke(state)
    return result["clarifying_questions"], result["context_summary"]


async def run_research(query: str, user_clarification: str, thread_id: str) -> tuple[str, str, float]:
    """Phase 2: full pipeline. Returns (report, email_status, score)."""
    config = {"configurable": {"thread_id": thread_id}}
    state  = _initial_state(query, user_clarification)
    result = await research_graph.ainvoke(state, config)
    return (
        result.get("report", ""),
        result.get("email_status", ""),
        result.get("report_score", 0.0),
    )

In [ ]:
# ── Gradio UI ──────────────────────────────────────────────────────────────────

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald"), title="Deep Research Agent") as demo:
    gr.Markdown(
        "# Deep Research Agent\n"
        "Enter a research topic. The agent clarifies scope, runs parallel web searches, "
        "checks sufficiency, writes and evaluates a report, then emails the result.\n\n"
        "**Pipeline:** Clarify → Plan → Search (parallel) → Sufficiency → Write → Evaluate → Email"
    )

    thread_state = gr.State(lambda: str(uuid.uuid4()))
    questions_state = gr.State([])

    with gr.Row():
        query_box   = gr.Textbox(label="Research topic", placeholder="e.g. What are the latest advances in fusion energy?", lines=2)
        clarify_btn = gr.Button("Get Scope Questions", variant="secondary")

    with gr.Group(visible=False) as clarify_group:
        gr.Markdown("### Scope questions — select or edit one, then add any extra context")
        questions_display = gr.Markdown()
        clarification_box = gr.Textbox(
            label="Your clarification (edit or rephrase one of the questions above)",
            placeholder="e.g. Focus on magnetic confinement (tokamak) advances in the past 2 years",
            lines=2,
        )
        extra_context_box = gr.Textbox(
            label="Additional context (optional)",
            placeholder="e.g. Include commercial projects and recent government funding",
            lines=2,
        )
        research_btn = gr.Button("Start Research", variant="primary", size="lg")

    with gr.Row():
        status_box = gr.Textbox(label="Pipeline status", lines=6, interactive=False,
                                placeholder="Status updates will appear here...")

    with gr.Row():
        with gr.Column(scale=2):
            report_box = gr.Markdown("The report will appear here when ready.")
        with gr.Column(scale=1):
            score_box  = gr.Textbox(label="Evaluator score", interactive=False)
            email_box  = gr.Textbox(label="Email status",    interactive=False)

    # Phase 1: clarify
    async def do_clarify(query, thread):
        if not query.strip():
            return gr.update(), "", "", [], thread
        try:
            questions, summary = await run_clarifier(query.strip())
            q_md = f"**{summary}**\n\n" + "\n\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))
            prefill = questions[0] if questions else ""
            return gr.update(visible=True), q_md, prefill, questions, str(uuid.uuid4())
        except Exception as exc:
            return gr.update(visible=True), f"Clarifier error: {exc}", "", [], thread

    clarify_btn.click(
        do_clarify,
        inputs=[query_box, thread_state],
        outputs=[clarify_group, questions_display, clarification_box, questions_state, thread_state],
    )
    query_box.submit(
        do_clarify,
        inputs=[query_box, thread_state],
        outputs=[clarify_group, questions_display, clarification_box, questions_state, thread_state],
    )

    # Phase 2: full research pipeline
    async def do_research(query, clarification, extra_context, thread):
        if not query.strip():
            return "Please enter a research topic.", "", "", ""
        full_clarification = clarification.strip()
        if extra_context.strip():
            full_clarification += f"\n\nAdditional context: {extra_context.strip()}"
        status = "Running: Plan → Search → Sufficiency check → Write → Evaluate → Email..."
        yield status, "", "", ""
        try:
            report, email_status, score = await run_research(query.strip(), full_clarification, thread)
            wc = len(report.split())
            status = f"Done. Report: {wc} words | Score: {score}/10 | Email: {email_status}"
            yield status, report, f"{score}/10", email_status
        except Exception as exc:
            yield f"Error: {exc}", "", "", ""

    research_btn.click(
        do_research,
        inputs=[query_box, clarification_box, extra_context_box, thread_state],
        outputs=[status_box, report_box, score_box, email_box],
    )

demo.launch(inbrowser=True)